# Model preparation + GPU inference (Google Colab)
**Colab-only part of the project.** Nothing is *trained* here: this notebook downloads the **pretrained** models (SpeechBrain ECAPA-TDNN speaker encoder, Coqui XTTS-v2), checks them, optionally runs enrollment + synthesis on a free GPU, and runs the ML experiments. Local inference with the Flask UI is described in the README.

* Use only voices you have permission to clone (your own recording or someone's explicit consent).
* XTTS-v2 weights are under the **Coqui Public Model License (non-commercial)**: https://coqui.ai/cpml
* Runtime -> Change runtime type -> **T4 GPU** (CPU also works, slowly).

In [ ]:
!nvidia-smi -L || echo "No GPU: synthesis will run on CPU (slow)" 

## 1. Get the project
Upload `speech_ai_project.zip` (the folder you downloaded) or mount Drive.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # choose speech_ai_project.zip
zipfile.ZipFile(next(iter(up))).extractall('.')
%cd speech_ai_project

## 2. Install (PyTorch is preinstalled on Colab)

In [ ]:
!pip -q install -r requirements.txt
!pip -q install coqui-tts speechbrain

## 3. Accept the XTTS licence and download the pretrained models

In [ ]:
import os
# Only after reading https://coqui.ai/cpml and agreeing:
os.environ['COQUI_TOS_AGREED'] = '1'
os.environ['TTS_DEVICE'] = 'auto'
!python scripts/prepare_models.py

## 4. Enroll two independent profiles from two reference recordings (~2 min each, same script)
Each profile gets its own folder under `profiles/`.

In [ ]:
print("Upload reference for PROFILE 1"); u1 = files.upload(); p1 = next(iter(u1))
print("Upload reference for PROFILE 2"); u2 = files.upload(); p2 = next(iter(u2))
!python scripts/cli.py enroll --profile "User 1" --wav "{p1}"
!python scripts/cli.py enroll --profile "User 2" --wav "{p2}" 

## 5. Generate new text in each enrolled voice

In [ ]:
!python scripts/cli.py synth --profile "User 1" --text "This sentence was never in the recording."
!python scripts/cli.py synth --profile "User 2" --text "A different sentence for the second speaker." 

In [ ]:
import glob, IPython.display as ipd
for slug in ('user-1', 'user-2'):
    runs = sorted(glob.glob(f'profiles/{slug}/outputs/*/final.wav'))
    if runs: print(slug); ipd.display(ipd.Audio(runs[-1]))

## 6. ML experiments: PCA + SVM, int8 quantisation
The quantisation report states exactly which layers were converted (ECAPA is convolutional; PyTorch dynamic int8 may convert none).

In [ ]:
!python scripts/cli.py classifier
!python scripts/cli.py quantize

## 7. Take the results home
Download `models/` (ECAPA + classifier) and optionally `profiles/` to reuse them locally.

In [ ]:
!zip -rq results.zip models profiles data/experiments
files.download('results.zip')